### Consigna del desafío 1

**1**. Vectorizar documentos. Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2**. Construir un modelo de clasificación por prototipos (tipo zero-shot). Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3**. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación
(f1-score macro) en el conjunto de datos de test. Considerar cambiar parámteros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial
y ComplementNB.

**4**. Transponer la matriz documento-término. De esa manera se obtiene una matriz
término-documento que puede ser interpretada como una colección de vectorización de palabras.
Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares. **La elección de palabras no debe ser al azar para evitar la aparición de términos poco interpretables, elegirlas "manualmente"**.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

# 20newsgroups por ser un dataset clásico de NLP ya viene incluido y formateado
# en sklearn
from sklearn.datasets import fetch_20newsgroups
import numpy as np

import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
import tqdm

In [6]:
# cargamos los datos (ya separados de forma predeterminada en train y test)
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

In [7]:
# instanciamos un vectorizador
# ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
tfidfvect = TfidfVectorizer()

# 1 Vectorizar Documentos

In [8]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)
y_train = newsgroups_train.target

In [9]:
def evaluar_similitud(ref_index, X, y, newsgroups_train, top_k=5):
    CANTIDAD_TEXTO = 100
    # Documento de referencia
    x_ref = X[ref_index].reshape(1, -1)
    ref_label = newsgroups_train.target_names[y[ref_index]]
    print("="*80)
    print(f"📌 Documento de referencia (idx={ref_index}) | Clase: {ref_label}")
    print("-"*80)
    print(newsgroups_train.data[ref_index][:CANTIDAD_TEXTO], "\n")

    # Calcular similitud coseno
    cossim = cosine_similarity(x_ref, X).ravel()
    # Obtener top_k más similares
    top_k_idx = np.argsort(-cossim)[:top_k]

    print("="*80)
    print(f"🔎 Top {top_k} documentos más similares:")
    print("="*80)

    for i, idx in enumerate(top_k_idx, start=1):
        label = newsgroups_train.target_names[y[idx]]
        print(f"[{i}] idx={idx} | Similitud={cossim[idx]:.4f} | Clase: {label}")
        print("-"*80)
        print(newsgroups_train.data[idx][:CANTIDAD_TEXTO], "\n")

    print("="*80)

In [10]:
evaluar_similitud(10,X_train,y_train, newsgroups_train, 5)

📌 Documento de referencia (idx=10) | Clase: rec.motorcycles
--------------------------------------------------------------------------------
I have a line on a Ducati 900GTS 1978 model with 17k on the clock.  Runs
very well, paint is the bro 

🔎 Top 5 documentos más similares:
[1] idx=10 | Similitud=1.0000 | Clase: rec.motorcycles
--------------------------------------------------------------------------------
I have a line on a Ducati 900GTS 1978 model with 17k on the clock.  Runs
very well, paint is the bro 

[2] idx=3543 | Similitud=0.4962 | Clase: rec.motorcycles
--------------------------------------------------------------------------------

Now you know why I am just a DOD member.  I like bikes and clubs but
the politics and other b*llsh* 

[3] idx=9171 | Similitud=0.4589 | Clase: rec.motorcycles
--------------------------------------------------------------------------------

More like those who use their backs instead of their minds to make
their living who are usually ign 

[

In [11]:
evaluar_similitud(20,X_train,y_train, newsgroups_train, 5)

📌 Documento de referencia (idx=20) | Clase: alt.atheism
--------------------------------------------------------------------------------

[...]

These don't seem like "little things" to me.  At least, they are orders
worse than the motto 

🔎 Top 5 documentos más similares:
[1] idx=20 | Similitud=1.0000 | Clase: alt.atheism
--------------------------------------------------------------------------------

[...]

These don't seem like "little things" to me.  At least, they are orders
worse than the motto 

[2] idx=10254 | Similitud=0.4432 | Clase: alt.atheism
--------------------------------------------------------------------------------


The "`little' things" above were in reference to Germany, clearly.  People
said that there were si 

[3] idx=1137 | Similitud=0.3694 | Clase: alt.atheism
--------------------------------------------------------------------------------


So, we should ban the ammunition?  Why not get rid of the guns?


It is worse than others?  The Na 

[4] idx=10779 | 

In [12]:
evaluar_similitud(653,X_train,y_train, newsgroups_train, 5)

📌 Documento de referencia (idx=653) | Clase: rec.sport.baseball
--------------------------------------------------------------------------------
Has anyone heard anything about Mel Hall this season?  I'd heard he wasn't
with the Yankees any more 

🔎 Top 5 documentos más similares:
[1] idx=653 | Similitud=1.0000 | Clase: rec.sport.baseball
--------------------------------------------------------------------------------
Has anyone heard anything about Mel Hall this season?  I'd heard he wasn't
with the Yankees any more 

[2] idx=4187 | Similitud=0.3219 | Clase: rec.sport.baseball
--------------------------------------------------------------------------------

Mel Hall signed with a Japanese team.
 

[3] idx=7394 | Similitud=0.2242 | Clase: rec.sport.baseball
--------------------------------------------------------------------------------

Mel is alive and well and playing in Japan. (The Yanks let him go because
he was asking for too muc 

[4] idx=5323 | Similitud=0.2020 | Clase: rec.spo

In [13]:
evaluar_similitud(7000,X_train,y_train, newsgroups_train, 5)

📌 Documento de referencia (idx=7000) | Clase: comp.sys.mac.hardware
--------------------------------------------------------------------------------
Dear Netters,

My sister has an Apple 12" Color Display hooked up to an LC.

Problem:  There is an a 

🔎 Top 5 documentos más similares:
[1] idx=7000 | Similitud=1.0000 | Clase: comp.sys.mac.hardware
--------------------------------------------------------------------------------
Dear Netters,

My sister has an Apple 12" Color Display hooked up to an LC.

Problem:  There is an a 

[2] idx=1533 | Similitud=0.1897 | Clase: soc.religion.christian
--------------------------------------------------------------------------------
*******
*******  This is somewhat long, but pleas read it!!!!!!!!!!!!!!!!!
*******



Boy am i glad  

[3] idx=386 | Similitud=0.1744 | Clase: sci.electronics
--------------------------------------------------------------------------------
Sci.E(E) netters:

I am setting out to build and market a small electronic device 

In [14]:
evaluar_similitud(8556,X_train,y_train, newsgroups_train, 5)

📌 Documento de referencia (idx=8556) | Clase: sci.crypt
--------------------------------------------------------------------------------


        But that's just the problem. There is no such thing as
        "MIME-Formatted". By analog 

🔎 Top 5 documentos más similares:
[1] idx=8556 | Similitud=1.0000 | Clase: sci.crypt
--------------------------------------------------------------------------------


        But that's just the problem. There is no such thing as
        "MIME-Formatted". By analog 

[2] idx=5976 | Similitud=0.3409 | Clase: sci.crypt
--------------------------------------------------------------------------------
1. Do a straight encryption of your keyrings and put the
        results with misleading names somew 

[3] idx=8478 | Similitud=0.1963 | Clase: talk.politics.mideast
--------------------------------------------------------------------------------


Peter,

I believe this is your most succinct post to date. Since you have nothing
to say, you say  

[4] idx=7

Al analizar los diferentes resultados, podemos notar como esta métrica de similitud nos puede ayudar a obtener una primera iteración en la detección de documentos similares. Para cada indice elegido (primer atributo de la función desarrollada) obtenemos los 5 documentos mas similares. Como era de esperarse, con esta configuración siempre el documento con mayor similitud será el mismo que se está evaluando, por lo cual se puede obtener un mejor resultado quitando este de la obtención de similitud.

A su vez, vemos como si bien suele estar acertada la descripción, muchas veces los documentos mas similares no pertenecen a la misma clase. Por ejemplo, la clase del primer documento (distinto del mismo docuemnto) en el indice 7000 no posee la misma clase: mientras que el documento es de la clase **comp.sys.mac.hardware**, el que presenta mayor similitud es de la clase **soc.religion.christian**

# 2 Clasificador Zero Shot

In [15]:
tfidfvect = TfidfVectorizer()

X_train = tfidfvect.fit_transform(newsgroups_train.data)
y_train = newsgroups_train.target

X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target

In [16]:
def zero_shot_predictor(X_train, X_test, y_train, y_test):

    similar_idx = cosine_similarity(X_test, X_train).argmax(axis=1)
    y_pred = y_train[similar_idx]
    
    f1_res = f1_score(y_test, y_pred, average='macro')

    print("f1 score:",f1_res)

    return y_pred

In [17]:
y_pred = zero_shot_predictor(X_train, X_test, y_train, y_test)

f1 score: 0.5049911553681621


Podemos observar como se realizó el clasificador zero shot de forma vectorizada: al utilizar arrays de numpy, podemos realizar todas las clasificaciones de forma simultanea. Al analizar los resultados, podemos determinar dos conclusiones:

1. El clasificador tiene mejor desempeño que una clasificador que adivina, llegando a un buen resultado para una primera iteración. Para 20 clases, si el clasificador estuviese adivinando tendríamos un f1 score de 0.05, mientras que nuestro valor es de 0.5. 
1. Esta metodología puede resultar algo ineficiente, ya que para realizar predicciones, deberiamos mantener el dataset completo de entrenamiento. Esto puede no resultar ideal para datasets de tamaño significativo.

# 3 Clasificador Naive Bayes

**3**. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación
(f1-score macro) en el conjunto de datos de test. Considerar cambiar parámteros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial
y ComplementNB.

Para este ejercicio, buscamos hacer una configuración que sea muy customizable, que nos permita encontrar el modelo e hiperparámetros con mejor desempeño. Utilizaremos para esta tarea OPTUNA

In [18]:
tfidfvect = TfidfVectorizer()

X_train = tfidfvect.fit_transform(newsgroups_train.data)
y_train = newsgroups_train.target

X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target

Definimos la función con rangos de parámetros de optuna en los cuales se entrenará:
- Alpha
- Fit Prior
- Modelo: **MultinomialNB** o **ComplementNB**

In [ ]:
# ----- Config -----
N_TRIALS = 40
SEED = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
SCORING = "f1_macro"

def objective(trial, model_type: str):
    # Espacio de búsqueda común
    alpha = trial.suggest_float("alpha", 1e-3, 10.0, log=True)
    fit_prior = trial.suggest_categorical("fit_prior", [True, False])

    if model_type == "mnb":
        model = MultinomialNB(alpha=alpha, fit_prior=fit_prior)
    elif model_type == "cnb":
        norm = trial.suggest_categorical("norm", [True, False])
        model = ComplementNB(alpha=alpha, fit_prior=fit_prior, norm=norm)
    else:
        raise ValueError("model_type must be 'mnb' or 'cnb'")

    # CV en train con F1 macro
    scores = cross_val_score(model, X_train, y_train, cv=CV, scoring=SCORING, n_jobs=-1)
    return scores.mean()


## MultinomialNB

Optimización de hiperparámetros

In [24]:
# ----- Optimización MNB -----
study_mnb = optuna.create_study(direction="maximize", study_name="MultinomialNB")
study_mnb.optimize(lambda t: objective(t, "mnb"), n_trials=N_TRIALS, show_progress_bar=False)
print("\n[MNB] Best CV F1-macro:", study_mnb.best_value)
print("[MNB] Best params:", study_mnb.best_params)

[I 2025-09-04 17:44:07,409] A new study created in memory with name: MultinomialNB
[I 2025-09-04 17:44:07,800] Trial 0 finished with value: 0.7570900849509676 and parameters: {'alpha': 0.00320820077001774, 'fit_prior': False}. Best is trial 0 with value: 0.7570900849509676.
[I 2025-09-04 17:44:08,163] Trial 1 finished with value: 0.7406411584206547 and parameters: {'alpha': 0.038220086832858574, 'fit_prior': True}. Best is trial 0 with value: 0.7570900849509676.
[I 2025-09-04 17:44:08,523] Trial 2 finished with value: 0.6586272254197254 and parameters: {'alpha': 0.5032322598158098, 'fit_prior': True}. Best is trial 0 with value: 0.7570900849509676.
[I 2025-09-04 17:44:08,908] Trial 3 finished with value: 0.7513831293129998 and parameters: {'alpha': 0.0010076997136311487, 'fit_prior': False}. Best is trial 0 with value: 0.7570900849509676.
[I 2025-09-04 17:44:09,278] Trial 4 finished with value: 0.7594938194422116 and parameters: {'alpha': 0.01176047665116436, 'fit_prior': False}. Best 


[MNB] Best CV F1-macro: 0.7599824757240727
[MNB] Best params: {'alpha': 0.007574562354855824, 'fit_prior': False}


Entrenamiento con mejores hiperparámetros y evaluación de resultados

In [25]:
best_mnb = MultinomialNB(**study_mnb.best_params).fit(X_train, y_train)
y_pred_mnb = best_mnb.predict(X_test)
print("[MNB] Test F1-macro:", f1_score(y_test, y_pred_mnb, average="macro"))

[MNB] Test F1-macro: 0.6877007516367813


## ComplementNB

Optimización de hiperparámetros

In [26]:
# ----- Optimización CNB -----
study_cnb = optuna.create_study(direction="maximize", study_name="ComplementNB")
study_cnb.optimize(lambda t: objective(t, "cnb"), n_trials=N_TRIALS, show_progress_bar=False)
print("\n[CNB] Best CV F1-macro:", study_cnb.best_value)
print("[CNB] Best params:", study_cnb.best_params)

[I 2025-09-04 17:44:22,755] A new study created in memory with name: ComplementNB
[I 2025-09-04 17:44:23,169] Trial 0 finished with value: 0.7247568148282355 and parameters: {'alpha': 0.0024309797866323684, 'fit_prior': False, 'norm': False}. Best is trial 0 with value: 0.7247568148282355.
[I 2025-09-04 17:44:23,439] Trial 1 finished with value: 0.7365325955973446 and parameters: {'alpha': 1.8288920988482318, 'fit_prior': False, 'norm': False}. Best is trial 1 with value: 0.7365325955973446.
[I 2025-09-04 17:44:23,820] Trial 2 finished with value: 0.7593710159458922 and parameters: {'alpha': 0.07405375294767669, 'fit_prior': True, 'norm': True}. Best is trial 2 with value: 0.7593710159458922.
[I 2025-09-04 17:44:24,216] Trial 3 finished with value: 0.746903150708125 and parameters: {'alpha': 0.02094338392879613, 'fit_prior': True, 'norm': True}. Best is trial 2 with value: 0.7593710159458922.
[I 2025-09-04 17:44:24,632] Trial 4 finished with value: 0.7450265052756551 and parameters: {'


[CNB] Best CV F1-macro: 0.7642207504544951
[CNB] Best params: {'alpha': 0.19519483291905618, 'fit_prior': False, 'norm': False}


Entrenamiento con mejores hiperparámetros y evaluación de resultados

In [27]:
# Entrenar y evaluar en test
best_cnb = ComplementNB(**study_cnb.best_params).fit(X_train, y_train)
y_pred_cnb = best_cnb.predict(X_test)
print("[CNB] Test F1-macro:", f1_score(y_test, y_pred_cnb, average="macro"))

[CNB] Test F1-macro: 0.6997851430118525


## Resultados

Podemos observar como mediante la optimización de hiperparámetros logramos obtener modelos que superen a nuestro clasificador Zero Shot. A su vez, la ventaja de utilizar un modelo entrenado radica en;
- Su tiempo de "entrenamiento" en significativamente menor al zero shot. Si bien el zero shot no se entrena, su predicción es singificativamente lenta, demora 3 s en ejecutar una predicción, mientras que logramos optimizar realizando 40 iteraciones de optuna en aproximadamente 18 s.
- Una vez entrenado el modelo de NB, no necesitamos mantener los datos de entrenamiento en memoria, siendo esto una ventaja notable para aplicaciones en producción  

# 4 Transponer la matriz documento-término

**4**. Transponer la matriz documento-término. De esa manera se obtiene una matriz
término-documento que puede ser interpretada como una colección de vectorización de palabras.
Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares. **La elección de palabras no debe ser al azar para evitar la aparición de términos poco interpretables, elegirlas "manualmente"**.

In [ ]:
X_train.shape #(documentos, palabras)

(11314, 101631)

In [28]:
X_transpose = X_train.T

In [31]:
X_transpose.shape #(palabras,documentos)

(101631, 11314)

In [47]:
def evaluar_similitud_palabras(name, X, tfidfvect, top_k=5):

    ref_index = tfidfvect.vocabulary_[name]
    idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

    # Documento de referencia
    x_ref = X[ref_index].reshape(1, -1)

    ref_label = name
    print("="*80)
    print(f"📌 Palabra de referencia (idx={ref_index}) | Palabra: {ref_label}")
    print("-"*80)
    # Calcular similitud coseno
    cossim = cosine_similarity(x_ref, X).ravel()
    # Obtener top_k más similares
    top_k_idx = np.argsort(-cossim)[:top_k]

    print("="*80)
    print(f"🔎 Top {top_k} palabras más similares:")
    print("="*80)

    for i, idx in enumerate(top_k_idx, start=1):
        label = idx2word[idx]
        print(f"[{i}] idx={idx} | Similitud={cossim[idx]:.4f} | Palabra: {label}")
        print("-"*80)
    print("="*80)

In [50]:
# es muy útil tener el diccionario opuesto que va de índices a términos
evaluar_similitud_palabras(name='car',X=X_transpose,tfidfvect=tfidfvect)

📌 Palabra de referencia (idx=25775) | Palabra: car
--------------------------------------------------------------------------------
🔎 Top 5 palabras más similares:
[1] idx=25775 | Similitud=1.0000 | Palabra: car
--------------------------------------------------------------------------------
[2] idx=25921 | Similitud=0.1797 | Palabra: cars
--------------------------------------------------------------------------------
[3] idx=30533 | Similitud=0.1770 | Palabra: criterium
--------------------------------------------------------------------------------
[4] idx=27562 | Similitud=0.1748 | Palabra: civic
--------------------------------------------------------------------------------
[5] idx=69218 | Similitud=0.1689 | Palabra: owner
--------------------------------------------------------------------------------


In [51]:
evaluar_similitud_palabras(name='truck',X=X_transpose,tfidfvect=tfidfvect)

📌 Palabra de referencia (idx=90357) | Palabra: truck
--------------------------------------------------------------------------------
🔎 Top 5 palabras más similares:
[1] idx=90357 | Similitud=1.0000 | Palabra: truck
--------------------------------------------------------------------------------
[2] idx=19054 | Similitud=0.3616 | Palabra: apologised
--------------------------------------------------------------------------------
[3] idx=73412 | Similitud=0.3474 | Palabra: profusely
--------------------------------------------------------------------------------
[4] idx=69963 | Similitud=0.3209 | Palabra: parlors
--------------------------------------------------------------------------------
[5] idx=85260 | Similitud=0.2945 | Palabra: steamed
--------------------------------------------------------------------------------


In [52]:
evaluar_similitud_palabras(name='god',X=X_transpose,tfidfvect=tfidfvect)

📌 Palabra de referencia (idx=43842) | Palabra: god
--------------------------------------------------------------------------------
🔎 Top 5 palabras más similares:
[1] idx=43842 | Similitud=1.0000 | Palabra: god
--------------------------------------------------------------------------------
[2] idx=52157 | Similitud=0.2688 | Palabra: jesus
--------------------------------------------------------------------------------
[3] idx=22698 | Similitud=0.2616 | Palabra: bible
--------------------------------------------------------------------------------
[4] idx=88519 | Similitud=0.2560 | Palabra: that
--------------------------------------------------------------------------------
[5] idx=38627 | Similitud=0.2548 | Palabra: existence
--------------------------------------------------------------------------------


In [53]:
evaluar_similitud_palabras(name='politics',X=X_transpose,tfidfvect=tfidfvect)

📌 Palabra de referencia (idx=72230) | Palabra: politics
--------------------------------------------------------------------------------
🔎 Top 5 palabras más similares:
[1] idx=72230 | Similitud=1.0000 | Palabra: politics
--------------------------------------------------------------------------------
[2] idx=48786 | Similitud=0.3139 | Palabra: iftccu
--------------------------------------------------------------------------------
[3] idx=46444 | Similitud=0.2634 | Palabra: hesh
--------------------------------------------------------------------------------
[4] idx=39581 | Similitud=0.2599 | Palabra: fascism
--------------------------------------------------------------------------------
[5] idx=23376 | Similitud=0.2548 | Palabra: bmwmoa
--------------------------------------------------------------------------------


In [55]:
evaluar_similitud_palabras(name='games',X=X_transpose,tfidfvect=tfidfvect)

📌 Palabra de referencia (idx=42606) | Palabra: games
--------------------------------------------------------------------------------
🔎 Top 5 palabras más similares:
[1] idx=42606 | Similitud=1.0000 | Palabra: games
--------------------------------------------------------------------------------
[2] idx=42595 | Similitud=0.2116 | Palabra: game
--------------------------------------------------------------------------------
[3] idx=81007 | Similitud=0.2057 | Palabra: scoring
--------------------------------------------------------------------------------
[4] idx=81298 | Similitud=0.1996 | Palabra: season
--------------------------------------------------------------------------------
[5] idx=75773 | Similitud=0.1655 | Palabra: rabbitball
--------------------------------------------------------------------------------


De forma similar al ejercicio numero 1, se observa como logramos analizar palabras similares, en base a los documentos en los que aparecen. Podemos ver para palabras mas generales(como games) como la similitud coseno suele ser mas baja, dando a entender que pueden tener varios contextos de usos. Para por ejemplo la palabra god, los terminos similares tienen una similitud coseno un poco mayor, la cual no disminuye tanto para los 5 mas similares. De ello puede intuirse que su contexto de uso de la palabra es un poco mas específico. 